In [1]:
from calc_flatfield import calc_flatfield

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/PHI_deadpix_maskcorrected.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/prefilter_combined_20250916_V20260626.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'

In [ ]:
flat_file, ghost_file = calc_flatfield(folder, 'temp',
                                       dark_file=dark_file,
                                       deadpix_file=deadpix_file,
                                       prefilter_file=prefilter_file,
                                       distortion_file=distortion_file)

start processing folder /home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/
found 9 files
reading and preprocessing the data


In [5]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

from utils import *
from scipy.ndimage import gaussian_filter
from astropy.io import fits
import glob

In [6]:
with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost)
flat_ = demodulate(flat)

In [7]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[1], vmin=-5e-3, vmax=5e-3)

In [8]:
files = sorted(glob.glob(folder + '*.fits.gz'))

In [9]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
wv = read_wavelengths(header)
#data = data.reshape(6,4,2048,2048)[cpos]
data = calc_continuum(data, wv, continuum=cpos)

data -= dark * 0.4
data /= flat
data = realign(data)
data = demodulate(data)

xr, yr = reflection_point_predict(header)
reflection = reflect(gaussian_filter(data[0], 8), xr, yr)

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(data[0])
plt.tight_layout()

In [11]:
plt.figure(figsize=(10,10))
plt.imshow(data[1] - ghost[1] * reflection, vmin=-30, vmax=30)
plt.tight_layout()